# 小规模 CBQM → QUBO Solver 验证

这本 notebook 验证完整的受约束问题链路：

```text
cbqm.v1
  → ProblemCase / TaskDefinition
  → compile_case_qubo
  → qubo.v1 solver
  → sample 投影回 CBQM
  → canonical feasibility / objective / exact 检查
```

重点不是只看 QUBO energy，而是确认 solver 结果回到原始业务约束后仍然正确。

## 1. 环境与导入

请选择项目 `.venv` kernel。下面会自动定位仓库根目录。

In [1]:
import json
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_cbqm, validate_qubo, validate_qubo_result
from lib.solvers.qubo import ExactQuboSolver, QaoaQuboSolver
from problem import (
    ProblemArtifact,
    ProblemCase,
    TaskDefinition,
    compile_case_qubo,
    evaluate_task_solution,
    project_qubo_sample_to_cbqm,
    solve_problem_task,
    validate_problem_case,
)
from tests.oracles import (
    enumerate_cbqm_feasible,
    enumerate_qubo,
    evaluate_cbqm_objective,
    is_cbqm_feasible,
    public_json_number,
)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

Project root: C:\Users\peter\Desktop\taiyi\QSolutionData
Python: c:\Users\peter\Documents\TaiyiQSolution\.venv\Scripts\python.exe


## 2. 定义一个 `cbqm.v1`

问题要求从三个资产中恰好选择一个。目标为最小化负 reward，因此 `asset_alpha` 是唯一最优可行选择。

In [2]:
cbqm = {
    "schema": "cbqm.v1",
    "problem_id": "notebook-small-cbqm",
    "variables": [
        {
            "index": 0,
            "name": "asset_alpha",
            "vartype": "BINARY",
            "kind": "decision",
            "metadata": {},
        },
        {
            "index": 1,
            "name": "asset_beta",
            "vartype": "BINARY",
            "kind": "decision",
            "metadata": {},
        },
        {
            "index": 2,
            "name": "asset_gamma",
            "vartype": "BINARY",
            "kind": "decision",
            "metadata": {},
        },
    ],
    "objective": {
        "sense": "minimize",
        "offset": 0.0,
        "linear": [[0, -3.0], [1, -2.0], [2, -1.0]],
        "quadratic": [[0, 1, 0.5], [1, 2, 0.25]],
    },
    "constraints": [
        {
            "name": "select_exactly_one",
            "family": "selection",
            "linear": [[0, 1.0], [1, 1.0], [2, 1.0]],
            "lower_bound": 1.0,
            "upper_bound": 1.0,
            "metadata": {},
        }
    ],
    "fixed_values": [],
    "metadata": {
        "purpose": "small constrained solver validation",
    },
}

validate_cbqm(cbqm)
print(json.dumps(cbqm, indent=2, ensure_ascii=False))

{
  "schema": "cbqm.v1",
  "problem_id": "notebook-small-cbqm",
  "variables": [
    {
      "index": 0,
      "name": "asset_alpha",
      "vartype": "BINARY",
      "kind": "decision",
      "metadata": {}
    },
    {
      "index": 1,
      "name": "asset_beta",
      "vartype": "BINARY",
      "kind": "decision",
      "metadata": {}
    },
    {
      "index": 2,
      "name": "asset_gamma",
      "vartype": "BINARY",
      "kind": "decision",
      "metadata": {}
    }
  ],
  "objective": {
    "sense": "minimize",
    "offset": 0.0,
    "linear": [
      [
        0,
        -3.0
      ],
      [
        1,
        -2.0
      ],
      [
        2,
        -1.0
      ]
    ],
    "quadratic": [
      [
        0,
        1,
        0.5
      ],
      [
        1,
        2,
        0.25
      ]
    ]
  },
  "constraints": [
    {
      "name": "select_exactly_one",
      "family": "selection",
      "linear": [
        [
          0,
          1.0
        ],
        [
          

## 3. 独立枚举 CBQM 可行域

先用 `tests.oracles` 中独立维护的精确枚举器计算原始 CBQM 可行域和 objective。它不经过 compiler，也不依赖任何 solver；notebook 不定义算法函数。

In [ ]:
feasible_rows = enumerate_cbqm_feasible(cbqm)
oracle_sample = feasible_rows[0]["sample"]
oracle_objective = feasible_rows[0]["objective_exact"]

for row in feasible_rows:
    print(row["sample"], "->", float(row["objective_exact"]))

assert oracle_sample == [1, 0, 0]
assert oracle_objective == -3
print("CBQM oracle:", oracle_sample, "objective =", float(oracle_objective))

## 4. 建立 `ProblemCase` 与 task

Solver 不直接消费 `ProblemCase`；它仍然只消费 `qubo.v1`。`ProblemCase` 负责记录原始 CBQM、派生 QUBO、task 语义和 best-known。

In [ ]:
source_artifact = ProblemArtifact(
    artifact_id="cbqm",
    representation="cbqm.v1",
    payload=cbqm,
)
task = TaskDefinition(
    task_id="select-one",
    canonical_artifact_id="cbqm",
    task_type="asset-selection",
    sense="minimize",
)
base_case = ProblemCase(
    problem_id=cbqm["problem_id"],
    artifacts=(source_artifact,),
    tasks=(task,),
    primary_artifact_id="cbqm",
)

case_report = validate_problem_case(base_case)
assert case_report.fully_checked
print("Artifacts:", [item.artifact_id for item in base_case.artifacts])
print("Tasks:", [item.task_id for item in base_case.tasks])
print("Case validation:", case_report.to_dict())

## 5. 编译为 `qubo.v1`

使用 quadratic penalty 编译器，并保留 compilation context、lineage 和完整性 hash。这里的 penalty 足够把不可行 assignment 推离最优点。

In [ ]:
compiler_config = {"default_penalty": 6.0}
compiled_case = compile_case_qubo(
    base_case,
    "cbqm",
    compiler_config,
    target_artifact_id="qubo-p6",
)
compiled_artifact = compiled_case.get_artifact("qubo-p6")
compiled_qubo = compiled_artifact.payload
validate_qubo(compiled_qubo)

compilation = compiled_artifact.transformation.context["compilation"]
print(json.dumps(compiled_qubo, indent=2, ensure_ascii=False))
print("Exact projection certified:", compilation["equivalence"]["exact_projection_certified"])
print("QUBO variable names:", compiled_qubo["variable_names"])
print("Slack variables:", compilation["slack_variables"])

## 6. 独立枚举编译后的 QUBO

在调用 solver 前，再独立枚举一次派生 QUBO。这一步为这个具体小模型确认 `default_penalty=6.0` 足够，并把 QUBO 最优 sample 投影回 CBQM 交叉检查。

In [ ]:
compiled_energy_table = enumerate_qubo(compiled_qubo)
compiled_oracle = compiled_energy_table[0]
compiled_oracle_projected = project_qubo_sample_to_cbqm(
    compiled_case,
    "qubo-p6",
    compiled_oracle["sample"],
)

assert compiled_oracle_projected == oracle_sample
assert is_cbqm_feasible(cbqm, compiled_oracle_projected)
assert evaluate_cbqm_objective(cbqm, compiled_oracle_projected) == oracle_objective

for row in compiled_energy_table:
    print(row["sample"], "->", float(row["energy_exact"]))
print("Compiled-QUBO oracle:", compiled_oracle)

## 7. 先直接验证派生 QUBO

这一步只证明 solver 对派生 QUBO 的结果正确。随后还必须投影回 CBQM，再检查原始约束和原始 objective。

In [ ]:
exact_solver = ExactQuboSolver()
raw_qubo_result = exact_solver.solve(
    compiled_qubo,
    config={"max_variables": 12},
)
validate_qubo_result(compiled_qubo, raw_qubo_result)

projected_sample = project_qubo_sample_to_cbqm(
    compiled_case,
    "qubo-p6",
    raw_qubo_result["best_sample"],
)
canonical_objective = evaluate_task_solution(
    compiled_case,
    "select-one",
    projected_sample,
)

assert raw_qubo_result["best_sample"] == compiled_oracle["sample"]
assert raw_qubo_result["best_energy"] == public_json_number(compiled_oracle["energy_exact"])
assert projected_sample == oracle_sample
assert evaluate_cbqm_objective(cbqm, projected_sample) == oracle_objective
assert canonical_objective == public_json_number(oracle_objective)
print("Raw QUBO result:", raw_qubo_result)
print("Projected CBQM sample:", projected_sample)
print("Canonical objective:", canonical_objective)

## 8. 使用 case-aware solver bridge

`solve_problem_task()` 把上面的步骤连成一个受验证流程。即使 solver 返回 `optimal`，bridge 仍会独立枚举所选小 QUBO，并检查编译证书、零 penalty、CBQM 可行性和 objective 映射，之后才允许 `exact_for_task=True`。

In [ ]:
exact_record = solve_problem_task(
    compiled_case,
    task_id="select-one",
    artifact_id="qubo-p6",
    solver=ExactQuboSolver(),
    config={"max_variables": 12},
    update_best=True,
    exact_verification_max_variables=12,
)

assert exact_record.canonical_solution == oracle_sample
assert evaluate_cbqm_objective(cbqm, exact_record.canonical_solution) == oracle_objective
assert exact_record.canonical_objective_value == public_json_number(oracle_objective)
assert exact_record.exact_for_task is True
assert exact_record.update is not None and exact_record.update.updated

updated_task = exact_record.case.get_task("select-one")
assert updated_task.best_known is not None
assert updated_task.best_known.exact is True

print("Canonical solution:", exact_record.canonical_solution)
print("Canonical objective:", exact_record.canonical_objective_value)
print("Exact for task:", exact_record.exact_for_task)
print("Best-known source:", updated_task.best_known.source)

## 9. 检查 exact evidence 与独立复核上限

`ExactQuboSolver.config['max_variables']` 控制 solver 是否允许运行；`exact_verification_max_variables` 控制应用层是否再次独立证明 optimality。它们是两道不同的安全门。

In [ ]:
exactness = updated_task.best_known.metadata["exactness"]
required_true_evidence = (
    "independent_qubo_optimality_verified",
    "compiler_exact_projection_certified",
    "compiler_certificate_consistent",
    "canonical_compilation_reproduced",
    "zero_penalty_verified",
    "objective_energy_identity_verified",
)
for field in required_true_evidence:
    assert exactness[field] is True, (field, exactness)
assert exactness["optimality_verification_reason"] == "search_space_exhausted"
assert validate_problem_case(exact_record.case).fully_checked

limited_record = solve_problem_task(
    compiled_case,
    task_id="select-one",
    artifact_id="qubo-p6",
    solver=ExactQuboSolver(),
    config={"max_variables": 12},
    update_best=False,
    exact_verification_max_variables=2,
)
assert limited_record.raw_result["status"] == "optimal"
assert limited_record.exact_for_task is False

print(json.dumps(exactness, indent=2, ensure_ascii=False))
print("Limited verification exact_for_task:", limited_record.exact_for_task)

## 10. 在同一个编译 QUBO 上运行 QAOA

QAOA 结果同样经过投影和 CBQM evaluator，但由于 solver status 是 `feasible`，即使命中 oracle，也不会升级成 exact。

In [ ]:
qaoa_record = solve_problem_task(
    compiled_case,
    task_id="select-one",
    artifact_id="qubo-p6",
    solver=QaoaQuboSolver(),
    config={
        "layers": 2,
        "optimizer_iterations": 20,
        "restarts": 4,
        "shots": 4096,
        "seed": 11,
        "max_variables": 12,
    },
    update_best=False,
)

assert qaoa_record.raw_result["status"] == "feasible"
assert is_cbqm_feasible(cbqm, qaoa_record.canonical_solution)
assert qaoa_record.exact_for_task is False

qaoa_objective_exact = evaluate_cbqm_objective(
    cbqm,
    qaoa_record.canonical_solution,
)
assert qaoa_record.canonical_objective_value == public_json_number(qaoa_objective_exact)
qaoa_gap = qaoa_objective_exact - oracle_objective
assert qaoa_gap >= 0

print("QAOA canonical sample:", qaoa_record.canonical_solution)
print("QAOA canonical objective:", qaoa_record.canonical_objective_value)
print("QAOA canonical gap:", float(qaoa_gap))
print("QAOA exact_for_task:", qaoa_record.exact_for_task)

## 11. 负例：solver 对 QUBO 正确，不代表对原 CBQM 正确

把 penalty 故意调得很小。Exact solver 仍然会正确找到这个弱 QUBO 的全局最优解，但该 assignment 可能违反原始 `select_exactly_one` 约束。canonical evaluator 必须把它拒绝。

In [ ]:
weak_case = compile_case_qubo(
    base_case,
    "cbqm",
    {"default_penalty": 0.1},
    target_artifact_id="qubo-weak",
)
weak_qubo = weak_case.get_artifact("qubo-weak").payload
weak_result = ExactQuboSolver().solve(
    weak_qubo,
    config={"max_variables": 12},
)
weak_projected = project_qubo_sample_to_cbqm(
    weak_case,
    "qubo-weak",
    weak_result["best_sample"],
)

canonical_error = None
try:
    evaluate_task_solution(weak_case, "select-one", weak_projected)
except ValueError as error:
    canonical_error = str(error)

assert canonical_error is not None
assert not is_cbqm_feasible(cbqm, weak_projected)
print("Weak-QUBO optimum:", weak_result["best_sample"])
print("Projected CBQM sample:", weak_projected)
print("Expected canonical rejection:", canonical_error)

## 12. 非恒等投影：恢复被 compiler 消元的 fixed variable

主例没有 slack 或 fixed variable，因此 source 与 QUBO 变量顺序恰好相同。下面追加一个独立微型案例：CBQM 有两个变量，其中 `policy_flag=1` 被 compiler 消元；solver 只返回一个 QUBO bit，投影必须恢复成两个 CBQM bits。

In [ ]:
fixed_cbqm = {
    "schema": "cbqm.v1",
    "problem_id": "notebook-fixed-projection",
    "variables": [
        {"index": 0, "name": "decision", "vartype": "BINARY"},
        {"index": 1, "name": "policy_flag", "vartype": "BINARY"},
    ],
    "objective": {
        "sense": "minimize",
        "offset": 0,
        "linear": [[0, -2], [1, 1]],
        "quadratic": [],
    },
    "constraints": [],
    "fixed_values": [{"index": 1, "value": 1}],
    "metadata": {},
}
validate_cbqm(fixed_cbqm)

fixed_case = ProblemCase(
    problem_id=fixed_cbqm["problem_id"],
    artifacts=(
        ProblemArtifact(
            artifact_id="cbqm",
            representation="cbqm.v1",
            payload=fixed_cbqm,
        ),
    ),
    tasks=(
        TaskDefinition(
            task_id="fixed-task",
            canonical_artifact_id="cbqm",
            sense="minimize",
        ),
    ),
)
fixed_case = compile_case_qubo(
    fixed_case,
    "cbqm",
    {"default_penalty": 4},
    target_artifact_id="qubo-fixed",
)
fixed_qubo = fixed_case.get_artifact("qubo-fixed")
fixed_context = fixed_qubo.transformation.context["compilation"]
fixed_raw = ExactQuboSolver().solve(
    fixed_qubo.payload,
    config={"max_variables": 8},
)
fixed_projected = project_qubo_sample_to_cbqm(
    fixed_case,
    "qubo-fixed",
    fixed_raw["best_sample"],
)

assert fixed_qubo.payload["num_variables"] == 1
assert fixed_context["fixed_values"] == [{"index": 1, "value": 1}]
assert fixed_raw["best_sample"] == [1]
assert fixed_projected == [1, 1]
assert evaluate_task_solution(fixed_case, "fixed-task", fixed_projected) == -1

print("QUBO sample:", fixed_raw["best_sample"])
print("Restored CBQM sample:", fixed_projected)
print("Fixed mapping:", fixed_context["fixed_values"])

## 结论

这本 notebook 同时验证了三种不同的“正确”：

1. **QUBO solver 正确**：结果满足 `qubo-result.v1`，energy 可从 QUBO 重算；
2. **编译与投影正确**：QUBO sample 能恢复 CBQM 顺序，包括被消元的 fixed variable；
3. **业务 task 正确**：恢复后的 sample 满足原始约束，objective 与独立 CBQM oracle 一致。

弱 penalty 负例说明：只比较 QUBO energy 会漏掉业务语义错误，因此 CBQM 校验不能省略。